## AgentCore Runtime의 Dynamic Client Registration

이 예제에서는 Dynamic Client Registration을 지원하는 MCP를 AgentCore Runtime에 배포하는 방법을 보여줍니다.

이 튜토리얼에서는 이 AgentCore 기능을 지원하는 Auth0 통합 예제를 생성합니다.

예를 들어 사용자의 social media login을 활용해 사용자를 동적으로 등록하고 MCP에 연결하도록 할 때 유용합니다.

이 튜토리얼에서는 다음 내용을 학습합니다.

* tool이 포함된 MCP server를 생성하는 방법
* server를 로컬에서 테스트하는 방법
* DCR을 지원하도록 Auth0 tenant를 구성하고 API와 app을 추가하는 방법
* Auth0의 DCR과 통합하여 server를 AWS에 배포하는 방법
* 배포된 server를 호출하는 방법

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Auth0에서 Tool 호스팅 + DCR                              |
| Tool 유형           | MCP server                                                |
| 튜토리얼 구성 요소  | AgentCore Runtime에 tool 호스팅, MCP server 생성         |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 중간                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 MCP Client         |

### 튜토리얼 아키텍처

이 튜토리얼에서는 이 예제를 AgentCore Runtime에 배포하는 방법을 설명합니다.

데모에서는 `add_numbers`, `multiply_numbers`, `greet_users` 세 가지 tool이 포함된 매우 간단한 MCP server를 사용합니다.

![Architecture](images/architecture.png)

### 튜토리얼 주요 기능

* MCP Server 호스팅
* Dynamic Client Registration (DCR)
* Auth0

In [ ]:
!pip install -Uq -r requirements.txt

**설치한 dependency가 반영되도록 kernel을 재시작하세요.**

## Auth0 구성

### 일반 설정

이 예제에서는 Auth0를 사용하여 DCR(Dynamic Client Registration)을 구현합니다. 시작하기 전에 Auth0 console에서 DCR 옵션을 활성화합니다.

- 왼쪽 panel menu에서 `Settings`를 클릭합니다. 그런 다음 **Tenant Settings** 아래에서 `Advanced` 옵션을 클릭합니다. *Dynamic Client Registration (DCR)* 옵션이 나타날 때까지 아래로 이동하여 *enable*합니다. 새 application을 생성할 때 connection도 활성화되도록 *Enable Application Connections*도 클릭합니다.

![Enable DCR](images/01_enable_dcr.png)

자세한 내용은 Auth0 [dynamic client registration](https://auth0.com/docs/get-started/applications/dynamic-client-registration) 문서를 참조하세요.

이제 기능을 활성화했으므로 *Google / Gmail* Identity를 MCP 사용자(app)의 login mechanism으로 사용합니다.

- 왼쪽 panel menu에서 `Authentication`, `Social`을 차례로 클릭합니다. **Social Connections** 화면에서 `google-oauth2` 옵션을 클릭합니다. 설정 화면에서 `Promote Connection to Domain Level` 옵션이 나타날 때까지 아래로 이동하여 활성화합니다.

![Enable Social](images/02_enable_social.png)

이제 Auth0 tenant의 모든 설정을 구성했습니다. 다음으로 Auth0 API를 생성합니다.

---

### API 생성

API를 생성합니다. 왼쪽 panel menu의 Applications 섹션에서 `APIs`를 클릭합니다. APIs 화면의 오른쪽 상단에서 `+ Create API`를 클릭합니다.

![API](images/03_create_api.png)

- Name: API 이름
- Identifier: API의 고유 identifier. 이 값은 authorization 호출에서 `audience` parameter로 사용됩니다.
- 다른 옵션은 기본값으로 유지하고 Create를 클릭합니다.

API 설정에서 `Permissions` tab을 클릭한 뒤 `<identifier_name>:Invoke` pattern에 따라 invoke 권한을 추가합니다.

![Permissions](images/04_permissions.png)

API 구성이 완료되었습니다. 이제 app을 생성합니다.

---

### Application 생성

application을 생성하여 구성을 마무리합니다. `Applications` 왼쪽 panel menu에서 `Applications`, `+ Create Application`을 차례로 클릭합니다.

app 생성 화면에서 app 이름을 지정하고 `Machine-to-Machine` 옵션을 선택한 뒤 create를 클릭합니다.

![App](images/05_app.png)

다음 화면에서 API를 선택하라는 메시지가 표시됩니다. 이전 단계에서 생성한 API를 선택하고 `all` button을 클릭하여 모든 작업을 허용한 뒤 `Authorize` button을 클릭합니다.

![Auth](images/06_authorizing.png)

이제 app이 생성되었습니다.

---

### App 정보 가져오기

예제에서 사용할 app 정보를 가져옵니다.

app을 클릭하고 `quickstart` page로 이동합니다. 이 page에는 필수 parameter가 모두 포함된 Linux `curl` 예제가 표시됩니다. 다음 예제와 같이 `--url` 값에서 domain을 복사합니다.

`--url` parameter:

```bash
--url https://<your-auth0-tenant>.us.auth0.com/oauth/token
```

이 Notebook에서 사용할 다음 환경 변수를 구성합니다.

```bash
export DISCOVERY_URL="https://<your-auth0-tenant>.us.auth0.com/.well-known/openid-configuration"
```

**여기를 실제 domain으로 교체하세요.**

In [ ]:
DISCOVERY_URL = "https://<your-auth0-tenant>.us.auth0.com/.well-known/openid-configuration"

audience도 입력해야 합니다. audience는 API identifier입니다. 이 예제를 따라 `ac-runtime-api`를 사용한다면 해당 값이 audience가 됩니다.

In [ ]:
# 다른 identifier로 생성했다면 교체
AUDIENCE = "ac-runtime-api"

### MCP Server 생성

이제 간단한 tool 세 개가 포함된 MCP server를 생성합니다.

In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers together"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """Greet a user by name"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### MCP Server 로컬 테스트(선택 사항)

원하는 경우 새 MCP server를 로컬에서 테스트할 수 있습니다.

다음 셀을 실행하여 테스트 Python script를 생성합니다.

In [ ]:
%%writefile test_my_mcp_client.py
import asyncio
from datetime import timedelta

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=timedelta(seconds=120), terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

MCP server를 로컬에서 테스트하려면 다음 단계를 수행합니다.

1. **Terminal 1**: MCP server 시작

```bash
python server.py
```

2. **Terminal 2**: 테스트 script 실행

```bash
python test_my_mcp_client.py
```

### AgentCore Runtime에 MCP Server 실행

제공된 URL과 audience를 사용하여 AgentCore Runtime app을 생성합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "mcp_dcr_sample"

auth_config = {
    "customJWTAuthorizer": {
        "allowedAudience": [AUDIENCE],
        "discoveryUrl": DISCOVERY_URL,
    }
}

response = agentcore_runtime.configure(
    entrypoint="server.py",
    auto_create_execution_role=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration=auth_config,
    protocol="MCP",
    memory_mode="NO_MEMORY",
    deployment_type="direct_code_deploy",
    runtime_type="PYTHON_3_13",
)

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

### 테스트

`mcp_auth0_client.py` 파일에는 AgentCore Runtime에 배포된 에이전트에 연결하는 로컬 구현이 포함되어 있습니다.

script를 실행하면 Google identity로 login하도록 요청하는 login page로 redirect됩니다.

login 후에는 email을 사용하여 app을 생성하도록 Auth0에 권한을 부여하라는 메시지가 표시됩니다.

![Redirect](images/07_redirect.png)

승인하면 성공 여부를 보여주는 일반 HTML page(`mcp_auth0_client.py`에 구현됨)로 redirect됩니다.

![OK](images/08_success.png)

script에서 필요한 환경 변수 설정

In [ ]:
agent_arn = launch_result.agent_arn
region = "us-west-2"
custom_endpoint = f"https://bedrock-agentcore.{region}.amazonaws.com"

agent_arn, custom_endpoint, AUDIENCE

In [ ]:
import mcp_auth0_client as mcp_client

await mcp_client.main(agent_arn, custom_endpoint, AUDIENCE)

## 리소스 정리

In [ ]:
agentcore_runtime.destroy()

### 🎉 축하합니다!

다음 작업을 성공적으로 완료했습니다.

- Auth0에 **tenant 생성**
- Auth0에서 **DCR 활성화** 및 API와 app 생성
- custom tool이 포함된 **MCP server** 생성
- MCP client를 사용하여 **로컬 테스트**
- **DCR 기반 인증** 설정
- **AgentCore Runtime**을 사용하여 AWS에 배포
- 적절한 인증으로 **원격 호출**
- MCP 개념 및 best practice **학습**

이제 MCP server가 Amazon Bedrock AgentCore Runtime에서 실행되고 있으며 production에서 사용할 준비가 되었습니다.